In [43]:
import pandas as pd

In [44]:
header=["seqid","source","type","start","end","score","strand","phase","attributes"]
in_file = "./TAIR10_GFF3_genes.gff"
gff = pd.read_csv(in_file, sep="\t", encoding='utf_8', names=header, header=None)

In [45]:
# remove protein ID from exon/CDS-parent
gff.loc[gff['type'] == 'CDS', 'attributes'] = gff.loc[gff['type'] == 'CDS', 'attributes'].str.split(',').str[0]
#gff.loc[gff['type'] == 'CDS', 'attributes'] = 'ID=' + gff.loc[gff['type'] == 'CDS', 'attributes']
gff.to_csv("./TAIR10_GFF3_genes_rm_prot.gff", header=None, sep="\t", index=None)
gff_reduced = gff.copy()

In [35]:
# replace critical type entries (manually tested beforehand)
allowed = ['chromosome', 'gene', 'mRNA', 'protein', 'exon', 'five_prime_UTR' 'CDS', 'three_prime_UTR']
keep = ['pseudogene', 'pseudogenic_transcript' 'pseudogenic_exon']
tomRNA = ['tRNA', 'ncRNA', 'mRNA_TE_gene', 'miRNA', 'snoRNA', 'rRNA', 'snRNA']
togene = ['transposable_element_gene']

In [36]:
# I guess, these are the ones, containing valid exons/CDS, but their respective parent is not recognized by PASA, resulting in missing IDs to refer to, either to their RNA or TE-gene.
# pseudogenes are not recognized, but since they only have pseudgenic transcripts and exons, the are also not checked, resulting in no missing ID references.
for type in tomRNA:
        gff.loc[gff['type'] == type, 'type'] = 'mRNA'
gff.loc[gff['type'] == 'transposable_element_gene', 'type'] = 'gene'
gff.to_csv("./TAIR10_GFF3_genes_crit.gff", header=None, sep="\t", index=None)

In [37]:
# replace all type entries
gff.loc[gff['type'] == 'pseudogene', 'type'] = 'gene'
gff.loc[gff['type'] == 'pseudogenic_transcript', 'type'] = 'mRNA'
gff.loc[gff['type'] == 'pseudogenic_exon', 'type'] = 'exon'
gff.to_csv("./TAIR10_GFF3_genes_full.gff", header=None, sep="\t", index=None)

In [46]:
# remove all types, which are not used by Kevins subset for annotation update (only protein coding genes)
rm_df = gff_reduced.loc[gff_reduced['type'].str.contains('tRNA|ncRNA|mRNA_TE_gene|miRNA|snoRNA|rRNA|snRNA|pseudogenic_transcript')==True]
rm_genes = rm_df['attributes'].str.split(';').str[1].str.split('Parent=').str[1].unique().tolist()
gff_reduced = gff_reduced[gff_reduced['attributes'].str.contains('|'.join(rm_genes)) == False]
gff_reduced = gff_reduced[(gff_reduced['type'] != 'chromosome') & (gff_reduced['type'] != 'protein')]
gff_reduced.to_csv("./TAIR10_GFF3_genes_reduced.gff", header=None, sep="\t", index=None)

In [ ]:
print(gff_reduced.head())